In [22]:
import pandas as pd
from pandas_datareader import data
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.datasets import fetch_california_housing

In [10]:
# Load dataset
housing = fetch_california_housing()


In [12]:
# Create Dataframe with features + target
df = pd.DataFrame(housing.data, columns = housing.feature_names)
df['MedHouseVal'] = housing.target    # target column

In [13]:
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


### Train Test Split :

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [18]:
# Initialize Decision Tree Regressor
DT_Reg = DecisionTreeRegressor(random_state=42)

In [19]:
# Train the model
DT_Reg.fit(X_train, y_train)


DecisionTreeRegressor(random_state=42)

In [20]:
# Predictions
y_pred = DT_Reg.predict(X_test)

In [27]:
# Evaluation
print("MAE:", metrics.mean_absolute_error(y_test, y_pred))   # MAE (Mean Absolute Error): average absolute difference between predicted & actual.
print("MSE:", metrics.mean_squared_error(y_test, y_pred))    # MSE (Mean Squared Error): average squared error (penalizes larger errors).
print("RMSE:", np.sqrt(metrics.mean_squared_error(y_test, y_pred)))    # RMSE (Root Mean Squared Error): square root of MSE, interpretable in original units.
print("R² Score:", r2_score(y_test, y_pred))    # R² Score: how much variance in target is explained by model (1 = perfect, 0 = bad).


MAE: 0.45467918846899225
MSE: 0.495235205629094
RMSE: 0.7037294974840077
R² Score: 0.622075845135081


In [28]:
# Hyperparameter tuning using GridSearchCV
param_grid = {
    'max_depth':[3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1,2,4],
}


**GridSearchCV** → finds the best parameters for Decision Tree.

- max_depth: maximum depth of the tree (limits complexity).

- min_samples_split: minimum samples needed to split a node.

- min_samples_leaf: minimum samples required in a leaf node.

In [29]:
grid_search = GridSearchCV (DT_Reg, param_grid, cv=5, scoring = 'r2', n_jobs= -1)
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [3, 5, 7, 10, None],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring='r2')

- cv=5: 5-fold cross-validation (train/test splits inside training set).

- scoring="r2": uses R² as performance measure.

- n_jobs=-1: use all CPU cores for speed.

In [31]:
print('Best Parameters:', grid_search.best_params_)
print('Best R² Score (CV): ', grid_search.best_score_)

Best Parameters: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2}
Best R² Score (CV):  0.7100830403599112
